# AgroTrace · Defectos de palta
Entrenamiento experimental de segmentación. Activa GPU y ejecuta en orden. No hay un modelo validado todavía. Las máscaras delimitan frutos completos y no porcentajes de lesión.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys, json
uploaded = files.upload()
assert len(uploaded) == 1, "Sube solamente agrotrace-defectos-colab.zip"
archive = next(iter(uploaded))
root = Path("/content/agrotrace")
assert not root.exists(), "Entorno ya preparado: reutiliza las celdas siguientes o reinicia el entorno"
with zipfile.ZipFile(archive) as z:
    for n in z.namelist():
        assert (root/n).resolve().is_relative_to(root.resolve()), "Ruta insegura"
    z.extractall(root)
os.chdir(root)
os.environ["PYTHONPATH"] = str(root/"ml")
sys.path.insert(0, str(root/"ml"))


In [ ]:
%pip install -q -r ml/requirements.txt
import torch
assert torch.cuda.is_available(), "Activa GPU en Entorno de ejecución > Cambiar tipo de entorno"
print(torch.cuda.get_device_name(0))


In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "agrotrace.prepare", "--zip", "dataset.zip", "--out", "dataset-preparado"], check=True)
from IPython.display import display, Image
display(Image(filename="dataset-preparado/masks-preview.jpg"))
audit = json.loads(Path("dataset-preparado/audit.json").read_text())
print(json.dumps(audit, indent=2, ensure_ascii=False))


## Revisión humana
Las fotos sospechosas se excluyen del entrenamiento por familia. El filtro no garantiza independencia completa por fruto o sesión. Cambia la variable solo después de inspeccionar las máscaras y el informe. No interpretes ausencia de detección como fruto sano.


In [ ]:
REVISADO = False
assert REVISADO, "Revisa la cuadrícula y audit.json antes de continuar"
subprocess.run([sys.executable, "-m", "agrotrace.train", "--data", "dataset-preparado/data.yaml", "--out", "runs/defectos-v1", "--epochs", "30", "--device", "0", "--reviewed"], check=True)


In [ ]:
manifest = json.loads(Path("runs/defectos-v1/model.json").read_text())
print(json.dumps(manifest, indent=2, ensure_ascii=False))
for name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    matches = list(Path("runs/defectos-v1").rglob(name))
    if matches:
        display(Image(filename=str(matches[0])))


## Test final (opcional)
Fija el modelo usando validación primero. Test se evalúa una sola vez para informar desempeño interno. Sus 24 imágenes no prueban generalización a planta.


In [ ]:
EVALUAR_TEST_FINAL = False
if EVALUAR_TEST_FINAL:
    subprocess.run([sys.executable, "-m", "agrotrace.evaluate", "--model-dir", "runs/defectos-v1", "--data", "dataset-preparado/data.yaml", "--device", "0"], check=True)


## Foto propia (opcional)
Una palta por fotografía, buena iluminación. La predicción es experimental y mantiene categoría comercial y mediciones no disponibles como nulas.


In [ ]:
PROBAR_FOTO = False
if PROBAR_FOTO:
    own = files.upload()
    assert len(own) == 1
    photo = next(iter(own))
    subprocess.run([sys.executable, "-m", "agrotrace.predict", "--model-dir", "runs/defectos-v1", "--image", photo, "--out", "runs/prueba-propia", "--device", "0"], check=True)
    display(Image(filename="runs/prueba-propia/analyzed.jpg"))


In [ ]:
import shutil
shutil.copy2("dataset-preparado/audit.json", "runs/defectos-v1/dataset-audit.json")
for name in ["README.dataset.txt", "README.roboflow.txt"]:
    shutil.copy2(Path("dataset-preparado")/name, Path("runs/defectos-v1")/name)
shutil.make_archive("agrotrace-modelo", "zip", "runs/defectos-v1")
files.download("agrotrace-modelo.zip")
